<a href="https://colab.research.google.com/github/mirmumin/netflow-crossdataset-ids/blob/main/notebooks/00_colab_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00 · Colab setup

Run once per Colab session. Put the four unzipped v3 folders (NF-UNSW-NB15-v3, NF-ToN-IoT-v3, NF-BoT-IoT-v3, NF-CICIDS2018-v3) **or** the four zip files in `MyDrive/thesis_data/` first.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!git clone https://github.com/mirmumin/netflow-crossdataset-ids.git /content/netflow-crossdataset-ids
%cd /content/netflow-crossdataset-ids
# Colab already has pandas, pyarrow, scikit-learn, pyyaml, scipy for this step.
# Installing the full pinned requirements.txt here would downgrade numpy and force a restart.
!pip install -q pyyaml pyarrow

Cloning into '/content/netflow-crossdataset-ids'...
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: '/content/netflow-crossdataset-ids'
/content


In [3]:
# Copy (or unzip) the data from Drive into data/raw/
import glob, os, shutil, zipfile
SRC = '/content/drive/MyDrive/thesis_data'
os.makedirs('data/raw', exist_ok=True)
for z in glob.glob(f'{SRC}/*.zip'):
    name = os.path.splitext(os.path.basename(z))[0]
    print('unzipping', z); zipfile.ZipFile(z).extractall(f'data/raw/{name}')
for d in glob.glob(f'{SRC}/NF-*'):
    if os.path.isdir(d):
        print('copying', d); shutil.copytree(d, f'data/raw/{os.path.basename(d)}', dirs_exist_ok=True)
!find data/raw -name '*.csv' -exec ls -lh {} \;

unzipping /content/drive/MyDrive/thesis_data/NF-BoT-IoT-v3.zip
unzipping /content/drive/MyDrive/thesis_data/NF-UNSW-NB15-v3.zip
unzipping /content/drive/MyDrive/thesis_data/NF-CICIDS2018-v3.zip
unzipping /content/drive/MyDrive/thesis_data/NF-ToN-IoT-v3.zip
-rw-r--r-- 1 root root 5.0G Nov  9 20:46 data/raw/NF-ToN-IoT-v3/02934b58528a226b_NFV3DATA-A11964_A11964/data/NF-ToN-IoT-v3.csv
-rw-r--r-- 1 root root 2.9K Nov  9 20:46 data/raw/NF-ToN-IoT-v3/02934b58528a226b_NFV3DATA-A11964_A11964/data/NetFlow_v3_Features.csv
-rw-r--r-- 1 root root 3.6G Nov  9 20:45 data/raw/NF-BoT-IoT-v3/d509c9db7490cf92_NFV3DATA-A11964_A11964/data/NF-BoT-IoT-v3.csv
-rw-r--r-- 1 root root 2.9K Nov  9 20:45 data/raw/NF-BoT-IoT-v3/d509c9db7490cf92_NFV3DATA-A11964_A11964/data/NetFlow_v3_Features.csv
-rw-r--r-- 1 root root 551M Nov  9 20:45 data/raw/NF-UNSW-NB15-v3/f7546561558c07c5_NFV3DATA-A11964_A11964/data/NF-UNSW-NB15-v3.csv
-rw-r--r-- 1 root root 2.9K Nov  9 20:45 data/raw/NF-UNSW-NB15-v3/f7546561558c07c5_NFV3DATA-

Copy the outputs of the last cell back to Claude. The processed parquet files are small; save them to Drive so later sessions skip this step:

In [6]:
import os, subprocess
from google.colab import userdata
t = userdata.get('GH_TOKEN')
r = subprocess.run(['git','clone',f'https://{t}@github.com/mirmumin/netflow-crossdataset-ids.git','/content/netflow-crossdataset-ids'], capture_output=True, text=True)
print((r.stdout + r.stderr).replace(t, '***'))
!mv /content/data/raw/* /content/netflow-crossdataset-ids/data/raw/
%cd /content/netflow-crossdataset-ids
!python scripts/prepare_data.py
!cat data/processed/schema_report.json
!head -40 data/processed/class_summary.csv
!mkdir -p /content/drive/MyDrive/thesis_data/processed && cp data/processed/* /content/drive/MyDrive/thesis_data/processed/

Cloning into '/content/netflow-crossdataset-ids'...

/content/netflow-crossdataset-ids
[unsw] subsampling NF-UNSW-NB15-v3.csv -> 500000 rows ...
[unsw] kept 502,822 rows, attack rate 0.062
[toniot] subsampling NF-ToN-IoT-v3.csv -> 500000 rows ...
[toniot] kept 503,336 rows, attack rate 0.394
[botiot] subsampling NF-BoT-IoT-v3.csv -> 500000 rows ...
[botiot] kept 499,675 rows, attack rate 0.996
[cic2018] subsampling NF-CICIDS2018-v3.csv -> 500000 rows ...
[cic2018] kept 503,152 rows, attack rate 0.140
Wrote class_summary.csv and schema_report.json to /content/netflow-crossdataset-ids/data/processed
[
  {
    "dataset": "NF-UNSW-NB15-v3",
    "n_cols": 55,
    "missing_v2": [],
    "v3_extra": [
      "FLOW_START_MILLISECONDS",
      "FLOW_END_MILLISECONDS",
      "SRC_TO_DST_IAT_MIN",
      "SRC_TO_DST_IAT_MAX",
      "SRC_TO_DST_IAT_AVG",
      "SRC_TO_DST_IAT_STDDEV",
      "DST_TO_SRC_IAT_MIN",
      "DST_TO_SRC_IAT_MAX",
      "DST_TO_SRC_IAT_AVG",
      "DST_TO_SRC_IAT_STDDEV"
    

In [7]:
!git pull
!python scripts/prepare_data.py --only botiot
!grep BoT data/processed/class_summary.csv
!cp data/processed/* /content/drive/MyDrive/thesis_data/processed/

remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 29 (delta 14), reused 22 (delta 10), pack-reused 0 (from 0)
Unpacking objects: 100% (29/29), 12.24 KiB | 1.75 MiB/s, done.
From https://github.com/mirmumin/netflow-crossdataset-ids
   25819dd..2f796a5  main       -> origin/main
Updating 25819dd..2f796a5
Fast-forward
 README.md                      |   5 +
 configs/default.yaml           |  13 ++
 notebooks/00_colab_setup.ipynb | 420 ++++++++++++++++++++++++++++++-----------
 scripts/prepare_data.py        |  16 +-
 scripts/run_all.sh             |   4 +-
 scripts/run_baselines.py       | 121 ++++++++++++
 scripts/summarise_baselines.py |  48 +++++
 src/data.py                    |  12 +-
 src/evaluate.py                |  25 +++
 src/models.py                  |  63 +++++++
 src/splits.py                  |  24 +++
 tests/test_data.py             |  23 +++
 12 files changed, 652 insertions(

In [9]:
!git pull
!python scripts/run_baselines.py --feature-sets v3
!python scripts/summarise_baselines.py
!cp -r results /content/drive/MyDrive/thesis_data/

remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 9 (delta 7), reused 9 (delta 7), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 2.01 KiB | 687.00 KiB/s, done.
From https://github.com/mirmumin/netflow-crossdataset-ids
   1c1598b..02c3087  main       -> origin/main
Updating 1c1598b..02c3087
Fast-forward
 scripts/make_toy_data.py |  3 +++
 scripts/run_baselines.py |  6 +++---
 scripts/run_fusion.py    | 12 ++++++------
 src/features.py          | 15 +++++++++++++++
 src/models.py            | 13 +++++--------
 5 files changed, 32 insertions(+), 17 deletions(-)
seed=42 v3     unsw   RF fit=  20.8s  F1: unsw=1.000 toniot=0.569 botiot=0.745 cic2018=0.246
seed=42 v3     unsw  XGB fit=  15.7s  F1: unsw=1.000 toniot=0.001 botiot=0.017 cic2018=0.034
seed=42 v3     unsw   LR fit=   8.4s  F1: unsw=0.991 toniot=0.550 botiot=0.045 cic2018=0.174
seed=42 v3     unsw  SVM fit=   2.0s  F1: unsw=0.997

In [10]:
!git pull
!pip install -q python-docx
!python scripts/make_report.py
!cp -r results /content/drive/MyDrive/thesis_data/

remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 8 (delta 5), reused 8 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 3.87 KiB | 1.29 MiB/s, done.
From https://github.com/mirmumin/netflow-crossdataset-ids
   02c3087..6db5d93  main       -> origin/main
Updating 02c3087..6db5d93
Fast-forward
 requirements.txt       |   1 +
 scripts/make_report.py | 145 +++++++++++++++++++++++++++++++++++++++++++++++++
 scripts/run_all.sh     |   1 +
 tests/test_data.py     |  12 ++++
 4 files changed, 159 insertions(+)
 create mode 100644 scripts/make_report.py
Wrote /content/netflow-crossdataset-ids/results/report.docx
